# 38 · Running many conversations at once
Level L4 · Part 5 · professional · ~35 min · builds on 14, 25, 35

## Goal
Run 50 Scout conversations at concurrency 1, 5 and 20 with `asyncio`, cap them with a semaphore and a
shared token-bucket rate limiter so the provider never answers 429, and report throughput and
p50/p95/p99 latency for each setting.

## The idea
One Scout conversation spends almost all its time waiting for the model. While it waits, Scout could
work on other conversations. `asyncio` (lesson 25) overlaps those waits. Two limits keep this safe. A
**semaphore** caps how many conversations run at once. A **token bucket** (lesson 14) caps how many
model calls per second leave your program, so the provider's own bucket never empties. Then you measure
two things: **throughput** (conversations finished per second) and **latency percentiles** (p50 is
the typical conversation, p95 and p99 the slow ones).

Analogy: a restaurant kitchen. The semaphore is the number of tables; the token bucket is how fast the
kitchen accepts orders. More tables help only until the kitchen is the bottleneck.

## Picture
```mermaid
flowchart LR
    Q["50 questions"] --> S{"semaphore<br/>N slots"}
    S --> C1["conversation"] & C2["conversation"] & C3["..."]
    C1 & C2 & C3 --> B{"shared token bucket<br/>45 calls/s"}
    B --> P["stub provider<br/>own bucket: 50 calls/s, burst 10"]
    P -->|"429 + retry-after"| C1
```

## Step by step
### 1. Setup from lesson 35 (shortened)
Scout's six notes, `search_notes`, the safe `calculator` and `run_tool`, shortened as in lesson 37.
This lesson needs no real model: every reply comes from a local stub, so no recorder is needed.

In [1]:
import ast, asyncio, json, math, operator, random, statistics, time

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]

The two tools: keyword search over the notes and the safe calculator.

In [2]:
def search_notes(query: str) -> list[dict]:
    words = query.lower().split()
    return [n for n in NOTES if all(w in n["text"].lower() for w in words)]

OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}
def evaluate(node):
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in OPS:
        return OPS[type(node.op)](evaluate(node.left), evaluate(node.right))
    raise ValueError("use numbers, + - * / and brackets, e.g. '24*60+39'")
def calculator(expression: str) -> float:
    return evaluate(ast.parse(expression, mode="eval").body)

print(search_notes("day on Mars"), calculator("24*60+39"))

[{'id': 3, 'text': 'A day on Mars lasts about 24 hours and 39 minutes.'}] 1479


The tool table and lesson 35's `run_tool`, which never raises. Schemas are kept to names only: the
stub below never reads them.

In [3]:
TOOLS = {"search_notes": search_notes, "calculator": calculator}
SCHEMAS = [{"name": name} for name in TOOLS]

def run_tool(call: dict, tools: dict) -> dict:
    fn = tools.get(call["name"])
    if fn is None:
        return {"content": f"unknown tool '{call['name']}'. Available: {', '.join(tools)}", "is_error": True}
    try:
        return {"content": json.dumps(fn(**call["args"])), "is_error": False}
    except Exception as e:
        return {"content": f"{type(e).__name__}: {e}", "is_error": True}

print(run_tool({"name": "calculator", "args": {"expression": "1/0"}}, TOOLS))

{'content': 'ZeroDivisionError: division by zero', 'is_error': True}


### 2. Fifty Scout questions
Every third question asks for the length of a Mars day in minutes, which needs a search and a
calculation; the others ask what a note says about a topic. The scripted reply for each step is written by hand
(it is not model output): search first, calculate if needed, then answer from the results.

In [4]:
TOPICS = [("Mars's moons", "moons"), ("Venus", "backwards"), ("Jupiter", "largest"),
          ("Perseverance", "Perseverance"), ("Saturn's rings", "rings")]
QUESTIONS = []
for i in range(50):
    if i % 3 == 0:
        QUESTIONS.append({"id": i, "text": f"Q{i:02d}: How many minutes long is a day on Mars?",
                          "query": "day on Mars", "math": True})
    else:
        label, query = TOPICS[i % 5]
        QUESTIONS.append({"id": i, "text": f"Q{i:02d}: What do my notes say about {label}?",
                          "query": query, "math": False})
BY_TEXT = {q["text"]: q for q in QUESTIONS}
print(len(QUESTIONS), "questions,", sum(q["math"] for q in QUESTIONS), "need the calculator")
print(QUESTIONS[0]["text"], "|", QUESTIONS[1]["text"])

50 questions, 17 need the calculator
Q00: How many minutes long is a day on Mars? | Q01: What do my notes say about Venus?


The script as a function of the message list, so a reply depends only on what Scout has seen.

In [5]:
def scripted_reply(q: dict, messages: list[dict]) -> dict:
    results = [json.loads(m["content"]) for m in messages if m["role"] == "tool"]
    call_id = f"q{q['id']}_call{len(results) + 1}"
    if not results:
        return {"role": "assistant", "tool_calls": [
            {"id": call_id, "name": "search_notes", "args": {"query": q["query"]}}]}
    if q["math"] and len(results) == 1:
        return {"role": "assistant", "tool_calls": [
            {"id": call_id, "name": "calculator", "args": {"expression": "24*60+39"}}]}
    if q["math"]:
        return {"role": "assistant", "text": f"A day on Mars is about {results[-1]:g} minutes."}
    return {"role": "assistant", "text": f"Note {results[0][0]['id']}: {results[0][0]['text']}"}

print(scripted_reply(QUESTIONS[0], [{"role": "user", "text": QUESTIONS[0]["text"]}]))

{'role': 'assistant', 'tool_calls': [{'id': 'q0_call1', 'name': 'search_notes', 'args': {'query': 'day on Mars'}}]}


### 3. The provider's token bucket
Lesson 14 said providers enforce limits with a **token bucket**. Here is one. It holds up to
`capacity` tokens and refills at `rate` tokens per second. Each request takes one token. An empty
bucket means "wait": `try_take` returns how many seconds until the next token. Passing `now` by hand
makes the demo exact.

In [6]:
class TokenBucket:
    def __init__(self, rate: float, capacity: int):
        self.rate, self.capacity = rate, capacity
        self.tokens, self.last = float(capacity), time.monotonic()

    def try_take(self, now: float | None = None) -> float:
        """Take one token and return 0.0, or return the seconds until a token is ready."""
        now = time.monotonic() if now is None else now
        self.tokens = min(self.capacity, self.tokens + (now - self.last) * self.rate)
        self.last = now
        if self.tokens >= 1:
            self.tokens -= 1
            return 0.0
        return (1 - self.tokens) / self.rate

A bucket of 50 per second with room for 10. Twelve requests in the same instant: ten pass, the next two
must wait. A tenth of a second later, five tokens have come back.

In [7]:
bucket = TokenBucket(rate=50, capacity=10)
t0 = bucket.last
print("burst of 12 :", [round(bucket.try_take(now=t0), 3) for _ in range(12)])
print("0.1 s later :", [round(bucket.try_take(now=t0 + 0.1), 3) for _ in range(6)])

burst of 12 : [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.02, 0.02]
0.1 s later : [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


This is why "50 per second" does not mean "any 50 requests in a second are fine": a burst of 12 at
once already fails.

### 4. A rate-limited stub provider
The stub plays the provider: it checks its own bucket (50 calls/s, burst 10), raises `RateLimited`
with a `retry_after` like a real 429, then waits a seeded, varied time and answers from the script.
About 1 call in 20 is slow (+0.4 s), like a real provider's tail.

In [8]:
class RateLimited(Exception):
    def __init__(self, retry_after: float):
        super().__init__(f"429 Too Many Requests, retry-after {retry_after:.3f}s")
        self.retry_after = retry_after

def stub_latency(qid: int, step: int) -> float:
    rng = random.Random(qid * 10 + step)              # same question, same step -> same delay
    return 0.04 + rng.expovariate(1 / 0.02) + (0.4 if rng.random() < 0.05 else 0.0)

class StubProvider:
    def __init__(self, rate: float = 50, burst: int = 10):
        self.bucket, self.calls, self.rejected = TokenBucket(rate, burst), 0, 0

    async def __call__(self, messages: list[dict], schemas: list[dict]) -> dict:
        wait = self.bucket.try_take()
        if wait > 0:
            self.rejected += 1
            raise RateLimited(wait)
        self.calls += 1
        q = BY_TEXT[messages[0]["text"]]
        step = sum(m["role"] == "assistant" for m in messages) + 1
        await asyncio.sleep(stub_latency(q["id"], step))   # waiting without blocking (lesson 25)
        return scripted_reply(q, messages)

### 5. Scout's loop, made async
Lesson 35's loop with three changes: it is `async def`, it awaits the model through `call_model`, and
it runs tools with `asyncio.to_thread` (lesson 25). `call_model` retries a 429 the lesson 14 way: wait
the server's `retry-after` plus jittered backoff, and give up after `max_attempts`. The optional
`limiter` comes in step 10.

In [9]:
class GiveUp(Exception):
    pass

def backoff_delay(attempt, base=0.05, cap=1.0, rng=random):        # lesson 14, smaller numbers
    return rng.uniform(0, min(cap, base * 2 ** attempt))

async def call_model(provider, messages, schemas, stats, limiter=None, max_attempts=8):
    for attempt in range(max_attempts):
        if limiter is not None:
            await limiter.acquire()                   # wait for OUR bucket before calling
        try:
            return await provider(messages, schemas)
        except RateLimited as e:
            stats["retries"] += 1
            await asyncio.sleep(e.retry_after + backoff_delay(attempt, rng=stats["rng"]))
    raise GiveUp(f"still rate limited after {max_attempts} attempts")

The loop itself. Printing is gone: fifty conversations would flood the screen.

In [10]:
async def run_agent_async(question, provider, tools, schemas, stats, limiter=None, max_steps=8):
    messages = [{"role": "user", "text": question}]
    for step in range(1, max_steps + 1):
        try:
            reply = await call_model(provider, messages, schemas, stats, limiter)
        except GiveUp as e:
            return {"answer": None, "stop": "rate_limited", "steps": step, "error": str(e)}
        messages.append(reply)
        calls = reply.get("tool_calls", [])
        if not calls:
            return {"answer": reply["text"], "stop": "end_of_turn", "steps": step}
        for call in calls:
            result = await asyncio.to_thread(run_tool, call, tools)
            messages.append({"role": "tool", "call_id": call["id"], "name": call["name"], **result})
    return {"answer": None, "stop": "step_limit", "steps": max_steps}

One conversation, timed. Jupyter already runs an event loop, so we `await` at the top level (lesson 25).

In [11]:
def new_stats(seed=38):
    return {"retries": 0, "rng": random.Random(seed), "in_flight": 0, "max_in_flight": 0}

start = time.perf_counter()
out = await run_agent_async(QUESTIONS[0]["text"], StubProvider(), TOOLS, SCHEMAS, new_stats())
print(out, f"| {time.perf_counter() - start:.2f} s")

{'answer': 'A day on Mars is about 1479 minutes.', 'stop': 'end_of_turn', 'steps': 3} | 0.20 s


Three model calls, about 0.2 s, almost all of it waiting.

### 6. A batch runner
`run_batch` runs every question through Scout with at most `concurrency` conversations at once. The
**semaphore** is the cap: `async with sem` waits until one of its slots is free. Each conversation's
latency is timed from when it gets its slot. A fresh provider per batch makes runs comparable.

In [12]:
async def run_batch(questions, concurrency, limiter=None, provider=None):
    provider, stats = provider or StubProvider(), new_stats()
    sem = asyncio.Semaphore(concurrency)

    async def one(q):
        async with sem:
            stats["in_flight"] += 1
            stats["max_in_flight"] = max(stats["max_in_flight"], stats["in_flight"])
            t0 = time.perf_counter()
            out = await run_agent_async(q["text"], provider, TOOLS, SCHEMAS, stats, limiter)
            out["latency"] = time.perf_counter() - t0
            stats["in_flight"] -= 1
            return out

    start = time.perf_counter()
    results = await asyncio.gather(*(one(q) for q in questions))
    return {"results": results, "wall": time.perf_counter() - start, "retries": stats["retries"],
            "rejected": provider.rejected, "calls": provider.calls, "max_in_flight": stats["max_in_flight"]}

### 7. Baseline: one at a time
Concurrency 1 is the plain `for` loop. Throughput is conversations finished per second of wall time.

In [13]:
base = await run_batch(QUESTIONS, concurrency=1)
answered = sum(r["stop"] == "end_of_turn" for r in base["results"])
print(f"answered {answered}/50 | model calls {base['calls']} | wall {base['wall']:.1f} s "
      f"| throughput {50 / base['wall']:.1f} conv/s | 429s {base['rejected']}")
print(base["results"][0]["answer"], "|", base["results"][1]["answer"])

answered 50/50 | model calls 117 | wall 8.8 s | throughput 5.7 conv/s | 429s 0
A day on Mars is about 1479 minutes. | Note 2: Venus spins backwards compared with most planets.


### 8. Percentiles, not averages
The **p95** is the latency that 95 % of conversations beat. We use the nearest-rank method: sort, then
take the value at position ceil(p/100 × n). With 50 samples, p99 is simply the slowest conversation;
a stable p99 needs hundreds of samples.

In [14]:
def percentile(values: list[float], p: float) -> float:
    ordered = sorted(values)
    return ordered[max(0, math.ceil(p / 100 * len(ordered)) - 1)]

lat = [r["latency"] for r in base["results"]]
print(f"mean {statistics.mean(lat):.3f} s")
for p in (50, 95, 99):
    print(f"p{p:<2}  {percentile(lat, p):.3f} s")

mean 0.176 s
p50  0.138 s
p95  0.514 s
p99  0.562 s


The mean sits above the p50 because a few slow calls (the +0.4 s tail) drag it up, yet it still
hides how bad the slowest ones are. Users remember the p95 and p99. One helper prints a row for every
experiment from now on.

In [15]:
def report(name: str, run: dict) -> dict:
    lat = [r["latency"] for r in run["results"]]
    row = {"setting": name, "wall_s": round(run["wall"], 2), "conv_per_s": round(50 / run["wall"], 1),
           **{f"p{p}": round(percentile(lat, p), 3) for p in (50, 95, 99)},
           "429s": run["rejected"], "failed": sum(r["stop"] != "end_of_turn" for r in run["results"])}
    print("  ".join(f"{k}={v}" for k, v in row.items()))
    return row

ROWS = [report("conc 1", base)]

setting=conc 1  wall_s=8.82  conv_per_s=5.7  p50=0.138  p95=0.514  p99=0.562  429s=0  failed=0


### 9. Concurrency 5 and 20 with only a semaphore
Same 50 questions, more slots. `max_in_flight` confirms the semaphore held.

In [16]:
for n in (5, 20):
    run = await run_batch(QUESTIONS, concurrency=n)
    ROWS.append(report(f"conc {n} no limiter", run))
    print(f"   max in flight {run['max_in_flight']} | retries {run['retries']}")

setting=conc 5 no limiter  wall_s=2.39  conv_per_s=21.0  p50=0.17  p95=0.524  p99=0.638  429s=49  failed=0
   max in flight 5 | retries 49


setting=conc 20 no limiter  wall_s=2.67  conv_per_s=18.7  p50=0.437  p95=2.206  p99=2.672  429s=208  failed=0
   max in flight 20 | retries 208


Concurrency 5 is several times faster, but the provider now rejects calls: bursts go above its
50 calls/s bucket. At 20 it gets worse: hundreds of 429s, a p99 of seconds, and some conversations
can give up after 8 attempts (see `failed`). Each 429 wastes a round trip and adds retry delay.

### 10. A shared token bucket on our side
The fix is to pace ourselves just below the provider: a client-side limiter at 45 calls/s with a burst
of 5. It wraps step 3's `TokenBucket`. The `asyncio.Lock` makes waiting callers queue in order, and
`asyncio.sleep` lets other conversations run while one waits for a token.

In [17]:
class AsyncRateLimiter:
    def __init__(self, rate: float, capacity: int):
        self.bucket, self.lock = TokenBucket(rate, capacity), asyncio.Lock()

    async def acquire(self) -> None:
        async with self.lock:
            while (wait := self.bucket.try_take()) > 0:
                await asyncio.sleep(wait)

One limiter, **shared** by every conversation in the batch, because the provider counts all of them
together.

In [18]:
for n in (5, 20):
    run = await run_batch(QUESTIONS, concurrency=n, limiter=AsyncRateLimiter(rate=45, capacity=5))
    ROWS.append(report(f"conc {n} + limiter", run))

setting=conc 5 + limiter  wall_s=2.61  conv_per_s=19.1  p50=0.207  p95=0.621  p99=0.658  429s=0  failed=0


setting=conc 20 + limiter  wall_s=2.72  conv_per_s=18.4  p50=0.848  p95=1.326  p99=1.355  429s=0  failed=0


### 11. Compare
Every run side by side. Exact numbers change a little between runs (the 429 count and the failures
most of all); the shape does not.

In [19]:
print(f"{'setting':<20}{'wall s':>7}{'conv/s':>7}{'p50':>7}{'p95':>7}{'p99':>7}{'429s':>6}{'failed':>7}")
for r in ROWS:
    print(f"{r['setting']:<20}{r['wall_s']:>7}{r['conv_per_s']:>7}{r['p50']:>7}{r['p95']:>7}"
          f"{r['p99']:>7}{r['429s']:>6}{r['failed']:>7}")
print(f"\nfloor set by our limiter: {base['calls']} calls / 45 per s = {base['calls'] / 45:.1f} s")

setting              wall s conv/s    p50    p95    p99  429s failed
conc 1                 8.82    5.7  0.138  0.514  0.562     0      0
conc 5 no limiter      2.39   21.0   0.17  0.524  0.638    49      0
conc 20 no limiter     2.67   18.7  0.437  2.206  2.672   208      0
conc 5 + limiter       2.61   19.1  0.207  0.621  0.658     0      0
conc 20 + limiter      2.72   18.4  0.848  1.326  1.355     0      0

floor set by our limiter: 117 calls / 45 per s = 2.6 s


Read it in three lines:
- **Throughput** rose from concurrency 1 to 5, then stopped: with the limiter both runs take about
  117 / 45 ≈ 2.6 s. The rate limit is the bottleneck now, not waiting.
- **Latency** per conversation went **up** at 20: conversations sit in the limiter's queue. More
  slots past the bottleneck buy only longer waits.
- **429s and failures** are zero with the shared limiter. It is a little slower than "conc 5 no
  limiter" (45 vs 50 calls/s) and far better than "conc 20 no limiter".

## At work
- **Bursts break limits, not averages.** Providers enforce limits with a token bucket (Anthropic: 60
  RPM "might be enforced as 1 request per second") and sudden ramps trigger "acceleration limits", so
  services cap concurrency and pace calls themselves. Sizing those caps for 5,000 users → lesson 56
- **Limits belong to the organisation or project, not the key** (Gemini counts per project), so every
  worker shares one budget. A limiter per process lets N processes send N times the rate; at company
  scale the limiter moves to a shared gateway. (Track 6.)
- **Cached input does not count towards Anthropic's input-token limit**, so at an 80 % cache hit
  2M ITPM carries 10M input tokens a minute. → lesson 41 (prompt caching)
- **Teams report p50/p95/p99, never only the mean**, and log latency per step so the slow tail can be
  traced. → lesson 49 (logging), lesson 55 (production Scout at 50 concurrent conversations)

## What just happened
- Scout's loop became `async`; 50 conversations shared one event loop and overlapped their waits.
- `asyncio.Semaphore(n)` capped conversations in flight; `max_in_flight` proved it.
- A `TokenBucket` modelled the provider's limit; the same class, behind an `asyncio.Lock`, paced our
  own calls just below it and removed every 429.
- Throughput rose until the rate limit became the bottleneck; past that point more concurrency only
  raised latency.
- Nearest-rank percentiles gave p50/p95/p99; with 50 samples p99 is the single worst run.

## Common mistakes
These three bugs are silent: nothing raises, the code is just slow or over the limit. So each wrong
cell checks the result with an `assert` and raises the error itself.

**1. A blocking call inside async code.** This provider uses `time.sleep` instead of `await
asyncio.sleep`. It freezes the whole event loop, so concurrency 10 is no faster than one at a time.
The `assert` makes the problem visible.

In [20]:
class BlockingProvider(StubProvider):
    async def __call__(self, messages, schemas):
        self.calls += 1
        q = BY_TEXT[messages[0]["text"]]
        step = sum(m["role"] == "assistant" for m in messages) + 1
        time.sleep(stub_latency(q["id"], step))               # blocks every conversation
        return scripted_reply(q, messages)

Run 10 questions one at a time, then all 10 at once on the blocking provider.

In [21]:
one_by_one = (await run_batch(QUESTIONS[:10], concurrency=1))["wall"]
blocked = (await run_batch(QUESTIONS[:10], concurrency=10, provider=BlockingProvider()))["wall"]
print(f"one at a time {one_by_one:.2f} s | 'concurrent' with time.sleep {blocked:.2f} s")
assert blocked < one_by_one / 2, "no speed-up: something is blocking the event loop"

one at a time 1.85 s | 'concurrent' with time.sleep 1.83 s


AssertionError: no speed-up: something is blocking the event loop

Fix: `await asyncio.sleep(...)` (or an async client) in async code, and `asyncio.to_thread` for plain
blocking functions such as Scout's tools.

In [22]:
fast = (await run_batch(QUESTIONS[:10], concurrency=10))["wall"]
print(f"with asyncio.sleep: {fast:.2f} s vs {one_by_one:.2f} s one at a time")

with asyncio.sleep: 0.73 s vs 1.85 s one at a time


**2. A semaphore created inside each task.** Each conversation gets its own fresh semaphore with 5
free slots, so nothing is shared and nothing is capped.

In [23]:
async def run_unbounded(questions, concurrency):
    stats = new_stats()
    async def one(q):
        async with asyncio.Semaphore(concurrency):           # new semaphore per task
            stats["in_flight"] += 1
            stats["max_in_flight"] = max(stats["max_in_flight"], stats["in_flight"])
            await run_agent_async(q["text"], StubProvider(rate=1000, burst=1000), TOOLS, SCHEMAS, stats)
            stats["in_flight"] -= 1
    await asyncio.gather(*(one(q) for q in questions))
    return stats["max_in_flight"]

peak = await run_unbounded(QUESTIONS[:20], concurrency=5)
assert peak <= 5, f"{peak} conversations ran at once, cap was 5"

AssertionError: 20 conversations ran at once, cap was 5

Fix: create the semaphore **once**, outside the tasks, as `run_batch` does.

In [24]:
print("max in flight with a shared semaphore:", (await run_batch(QUESTIONS[:20], concurrency=5))["max_in_flight"])

max in flight with a shared semaphore: 5


**3. A limiter per conversation.** Each conversation paces itself at 45 calls/s, but the provider sees
all of them together, so the 429s come back.

In [25]:
async def run_per_task_limiter(questions, concurrency):
    provider, stats, sem = StubProvider(), new_stats(), asyncio.Semaphore(concurrency)
    async def one(q):
        async with sem:
            await run_agent_async(q["text"], provider, TOOLS, SCHEMAS, stats, AsyncRateLimiter(45, 5))
    await asyncio.gather(*(one(q) for q in questions))
    return provider.rejected

rejected = await run_per_task_limiter(QUESTIONS, concurrency=20)
assert rejected == 0, f"provider rejected {rejected} calls"

AssertionError: provider rejected 230 calls

Fix: one limiter for everything that shares the provider's limit, passed into every conversation.

In [26]:
shared = await run_batch(QUESTIONS, concurrency=20, limiter=AsyncRateLimiter(45, 5))
print("429s with one shared limiter:", shared["rejected"])

429s with one shared limiter: 0


## Try it
Find the **knee**: run all 50 questions with the shared limiter (45 calls/s, burst 5) at concurrency
1, 2, 5, 10, 20 and 40. Print throughput and p95 for each, then print the smallest concurrency whose
throughput is within 10 % of the best. Why is that a better setting than 40?
Hint: keep each (concurrency, throughput) pair in a list, then take the max.

In [27]:
# your code here

## Key terms
- **concurrency limit** — the most conversations (or calls) allowed in flight at once; here enforced by a semaphore.
- **semaphore (`asyncio.Semaphore`)** — a counter of free slots: `async with sem` takes a slot, waits if none is free, and gives it back on exit.
- **client-side rate limiter** — a token bucket in your own code, shared by every caller, that paces requests just below the provider's limit so it never returns 429.
- **throughput** — work finished per unit of time, here conversations per second of wall time.
- **p50 / p95 / p99** — latency percentiles: the time that 50 %, 95 % or 99 % of requests finish within.
- **tail latency** — the slow end of the latency distribution (p95, p99), which the mean hides.